# Real-Time Credit Scoring — Hybrid Table + REST API (Showcase)

Notebook ini men-*showcase* alur **real-time decision engine** CLIK, dijalankan **di dalam Snowflake Notebook**:

1. **Step 3 — Point lookup by PK ke Hybrid Table** (`SUBJECT_FEATURES_ENCODED_HT`, Unistore) untuk ambil 60 fitur ter-encode.
2. **Step 4 — Call Model Service via REST API** (SPCS ingress) dari dalam notebook via **External Access Integration**.
3. **Step 5 — Mapping bisnis**: PD → credit score → decision.

### Prasyarat (WAJIB)
- Service `CLIK_PD_SERVICE` sudah **READY** (lihat `04b_deploy_service.py`).
- Hybrid table `SUBJECT_FEATURES_ENCODED_HT` sudah terisi.
- External Access Integration **`CLIK_SPCS_EAI`** + secret `CLIK_PD_PAT` sudah dibuat (lihat `04b_notebook_rest_setup.sql`).
- **Attach EAI ke notebook ini:** menu `⋯` → **Notebook settings** → **External access integrations** → aktifkan `CLIK_SPCS_EAI` → restart.

In [ ]:
# Cell 1 - Setup: session, ingress URL, PAT dari secret
import json
import time
import requests
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()

# Ingress URL diambil dinamis dari service (tanpa hardcode)
ep = session.sql("SHOW ENDPOINTS IN SERVICE CLIK_WORKSHOP2.PUBLIC.CLIK_PD_SERVICE").collect()
INGRESS_URL = ep[0]["ingress_url"]
ENDPOINT_URL = f"https://{INGRESS_URL}/predict-proba"

# PAT: di Snowflake Notebook, secret diakses via st.secrets (WAJIB attach secret
# ke notebook: Notebook settings > External access > tambahkan CLIK_PD_PAT, atau
# ALTER NOTEBOOK <nb> SET SECRETS=('CLIK_PD_PAT'=CLIK_WORKSHOP2.PUBLIC.CLIK_PD_PAT)).
# Fallback: getpass bila dijalankan di luar Snowsight.
try:
    import streamlit as st
    PAT_TOKEN = st.secrets["CLIK_PD_PAT"]
    print("PAT dibaca dari st.secrets['CLIK_PD_PAT'].")
except Exception as e:
    import getpass
    print(f"st.secrets tidak tersedia ({type(e).__name__}). Masukkan PAT manual.")
    PAT_TOKEN = getpass.getpass("Masukkan PAT: ")
HEADERS = {
    "Authorization": f'Snowflake Token="{PAT_TOKEN}"',
    "Content-Type": "application/json",
}
print("Endpoint:", ENDPOINT_URL)

## Step 3 — Point Lookup dari Hybrid Table (Unistore)

Orkestrasi menerima `Subject ID` lalu mengambil 60 fitur ter-encode via **point lookup by primary key** ke hybrid table `SUBJECT_FEATURES_ENCODED_HT` — pola *precalculated feature table* pada arsitektur real-time.

In [ ]:
# Cell 2 - Step 3: point lookup fitur dari Hybrid Table
SUBJECT_IDS = ['SUBJ000000020', 'SUBJ000000044', 'SUBJ000000068']
placeholders = ", ".join([f"'{s}'" for s in SUBJECT_IDS])

t0 = time.perf_counter()
df = session.sql(f"""
    SELECT * EXCLUDE (SUBJECT_ID)
    FROM CLIK_WORKSHOP2.PUBLIC.SUBJECT_FEATURES_ENCODED_HT
    WHERE SUBJECT_ID IN ({placeholders})
""").to_pandas()
lookup_ms = (time.perf_counter() - t0) * 1000.0
print(f"[Hybrid Table Lookup] {df.shape[0]} baris x {df.shape[1]} kolom dalam {lookup_ms:.0f} ms (point lookup by PK)")
df.head()

## Step 4 — Call Model Service via REST API

POST fitur ke endpoint SPCS. Payload `dataframe_split` **wajib** menyertakan key `index` → gunakan `df.to_json(orient="split")` (jangan `index=False`).

In [ ]:
# Cell 3 - Step 4: call Model Service via REST API (dari dalam notebook, lewat EAI)
split_obj = json.loads(df.to_json(orient="split"))
payload = {"dataframe_split": split_obj}

t0 = time.perf_counter()
resp = requests.post(ENDPOINT_URL, headers=HEADERS, json=payload, timeout=30)
infer_ms = (time.perf_counter() - t0) * 1000.0
print(f"[Model Service REST] HTTP {resp.status_code} dalam {infer_ms:.0f} ms")
resp.raise_for_status()
result = resp.json()
print(json.dumps(result, indent=2)[:600])

## Step 5 — Mapping Bisnis (PD → Credit Score → Decision)

Output service: `{"data": [[idx, {..., PREDICT_PROBA_1}]]}`. `PREDICT_PROBA_1` = P(default) = PD score.

In [ ]:
# Cell 4 - Step 5: mapping bisnis
import pandas as pd

def extract_pd(row_out):
    obj = row_out[1] if isinstance(row_out, list) else row_out
    return float(obj["PREDICT_PROBA_1"]) if isinstance(obj, dict) else float(row_out[-1])

rows = []
for sid, row_out in zip(SUBJECT_IDS, result["data"]):
    pd_score = extract_pd(row_out)
    credit_score = 300 + round(550 * (1 - pd_score))
    decision = "APPROVE" if pd_score < 0.10 else ("REVIEW" if pd_score < 0.30 else "REJECT")
    rows.append({"SUBJECT_ID": sid, "PD": round(pd_score, 4), "CREDIT_SCORE": credit_score, "DECISION": decision})

result_df = pd.DataFrame(rows)
print(f"Total latency (lookup + REST): {lookup_ms + infer_ms:.0f} ms")
result_df